<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/NLP/blob/main/Sentiment_Analysis_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df["sentiment"] = df["sentiment"].map({
    "positive":1,
    "negative":0
})

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X_train)

In [6]:
X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test)

In [7]:
print(X_train.iloc[0])
print(X_train_seq[0])

Remember the wooden, undramatic literary adaptations of the 1970s at their worst? You will when you see this broadly acted, unintentionally hilarious piece of chocolate-box adaptation. Most culpable of all is Catherine Z-J who, while undeniably easy on the eye, substitutes swishing a big dress and looking sultry for actually turning in a performance. Played po-faced like a melodrama, or Cold Comfort Farm without the jokes, this effort is not helped by a scriptwriter with a tin ear for dialogue who misses entirely the novel's sense of irony or tragedy. A shame, given the quality of the acting talent on offer - Joan Plowright, Claire Skinner, Steven Macintosh all deserve better than this.
[380, 1, 1781, 4652, 5508, 4, 1, 3322, 30, 61, 251, 21, 80, 52, 21, 63, 10, 893, 2983, 641, 353, 4, 8377, 842, 1416, 88, 4, 29, 6, 2583, 4653, 1629, 35, 133, 5910, 746, 20, 1, 843, 7554, 3, 180, 2584, 2, 282, 7555, 17, 167, 1654, 8, 3, 235, 247, 2325, 37, 3, 2267, 39, 1004, 4403, 2892, 198, 1, 630, 10, 

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post"
)

In [9]:
print(X_train_pad.shape)

(3999, 200)


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=10000,
              output_dim=64,
              input_length=max_length),

    LSTM(64),

    Dense(1, activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [27]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [23]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 29s 522ms/step - accuracy: 0.5114 - loss: 0.6931 - val_accuracy: 0.5475 - val_loss: 0.6903
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 39s 479ms/step - accuracy: 0.5946 - loss: 0.6716 - val_accuracy: 0.5425 - val_loss: 0.6728
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 491ms/step - accuracy: 0.6590 - loss: 0.5773 - val_accuracy: 0.5688 - val_loss: 0.6783
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 500ms/step - accuracy: 0.6899 - loss: 0.4737 - val_accuracy: 0.5375 - val_loss: 0.7241
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 42s 521ms/step - accuracy: 0.7271 - loss: 0.4173 - val_accuracy: 0.5238 - val_loss: 0.9385
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 490ms/step - accuracy: 0.7343 - loss: 0.3977 - val_accuracy: 0.5213 - val_loss: 0.8381
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 42s 509ms/step - accuracy: 0.7580 - loss: 0.3835 - val_accuracy: 0.5650 - val_loss: 0.8409
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 502ms/step - accuracy: 0.7896 - loss: 0.3635 - val_accu

In [28]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Accuracy:", accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.5130 - loss: 1.0908
Test Accuracy: 0.5130000114440918


In [26]:
new_reviews = [
    "The movie was absolutely amazing.",
    "I wasted my time watching this film."
]

new_seq = tokenizer.texts_to_sequences(new_reviews)
new_pad = pad_sequences(new_seq, maxlen=max_length, padding="post")

predictions = model.predict(new_pad)

for review, pred in zip(new_reviews, predictions):
    sentiment = "Positive" if pred > 0.5 else "Negative"
    print(f"{review} --> {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
The movie was absolutely amazing. --> Positive
I wasted my time watching this film. --> Positive
